# Layer-1 IndoBERT Token Classifier (Colab)

Notebook ini adalah versi Colab dari `layer1_indobert_token_classifier.py`.

Urutan pakai:
1. Jalankan sel **Install dependencies**.
2. Atur `MODEL_DIR` dan input inferensi.
3. Jalankan sel inferensi Layer 1 (single atau batch).
4. Jalankan sel Layer 2a/2b untuk konversi fonem.

Catatan: notebook ini mengharuskan runtime GPU Colab (CUDA) dan akan otomatis memakai GPU.

In [ ]:
# Jika runtime Colab baru, jalankan dulu.
!python -m pip install -q torch transformers g2p-id g2p-en nltk

In [ ]:
from functools import lru_cache
import re
from typing import NamedTuple


# Definisikan struktur TokenLabel
class TokenLabel(NamedTuple):
    token: str
    label: str
    score: float


def _import_layer2_deps():
    try:
        import nltk  # type: ignore
        from g2p_id import G2P  # type: ignore
        from g2p_en import G2p  # type: ignore
    except ModuleNotFoundError as err:
        raise ModuleNotFoundError(
            "Dependensi Layer 2 belum lengkap. Install dulu: python -m pip install g2p-id g2p-en nltk"
        ) from err
    return nltk, G2P, G2p


def _ensure_g2p_en_resources(nltk) -> None:
    packages = {
        "averaged_perceptron_tagger": "taggers/averaged_perceptron_tagger",
        "cmudict": "corpora/cmudict",
        "punkt": "tokenizers/punkt",
    }
    for pkg, path in packages.items():
        try:
            nltk.data.find(path)
        except LookupError:
            nltk.download(pkg, quiet=True)


nltk, G2P_ID_CLASS, G2P_EN_CLASS = _import_layer2_deps()
_ensure_g2p_en_resources(nltk)

g2p_id_model = G2P_ID_CLASS()
g2p_en_model = G2P_EN_CLASS()

ALPHABET_EJA_MAP = {
    "a": "a", "b": "be", "c": "ce", "d": "de", "e": "e", "f": "ef", "g": "ge",
    "h": "ha", "i": "i", "j": "je", "k": "ka", "l": "el", "m": "em", "n": "en",
    "o": "o", "p": "pe", "q": "ki", "r": "er", "s": "es", "t": "te", "u": "u",
    "v": "ve", "w": "we", "x": "eks", "y": "ye", "z": "zet"
}

DISMV_SHORT_PATTERN = "v o n"
DISMV_LONG_PATTERN = "v o n e m"

ARPABET_TO_IPA = {
    "AA": "ɑ", "AE": "æ", "AH": "ʌ", "AO": "ɔ", "AW": "aʊ", "AY": "aɪ",
    "B": "b", "CH": "tʃ", "D": "d", "DH": "ð", "EH": "ɛ", "ER": "ɝ",
    "EY": "eɪ", "F": "f", "G": "ɡ", "HH": "h", "IH": "ɪ", "IY": "i",
    "JH": "dʒ", "K": "k", "L": "l", "M": "m", "N": "n", "NG": "ŋ",
    "OW": "oʊ", "OY": "ɔɪ", "P": "p", "R": "r", "S": "s", "SH": "ʃ",
    "T": "t", "TH": "θ", "UH": "ʊ", "UW": "u", "V": "v", "W": "w",
    "Y": "j", "Z": "z", "ZH": "ʒ"
}


def arpabet_to_ipa_sequence(tokens: list[str]) -> str:
    ipa_tokens: list[str] = []
    for token in tokens:
        base = re.sub(r"\d", "", token).upper()
        if not base:
            continue
        ipa_tokens.append(ARPABET_TO_IPA.get(base, base.lower()))
    return " ".join(ipa_tokens)


def _coerce_g2p_output_to_str(raw) -> str:
    if isinstance(raw, tuple):
        return _coerce_g2p_output_to_str(raw[0])
    if isinstance(raw, list):
        if raw and isinstance(raw[0], str):
            return " ".join(part for part in raw if part)
        return " ".join(str(part) for part in raw if part)
    return str(raw)


def _g2p_id_convert(word: str) -> str:
    if hasattr(g2p_id_model, "to_phoneme"):
        return _coerce_g2p_output_to_str(g2p_id_model.to_phoneme(word))
    if hasattr(g2p_id_model, "predict"):
        return _coerce_g2p_output_to_str(g2p_id_model.predict(word))
    if hasattr(g2p_id_model, "transcribe"):
        return _coerce_g2p_output_to_str(g2p_id_model.transcribe(word))
    if callable(g2p_id_model):
        return _coerce_g2p_output_to_str(g2p_id_model(word))
    raise AttributeError(
        "Objek g2p-id tidak punya API yang didukung (to_phoneme/predict/transcribe/__call__)."
    )


@lru_cache(maxsize=8192)
def g2p_id_phoneme(word: str) -> str:
    return _g2p_id_convert(word)


@lru_cache(maxsize=8192)
def g2p_foreign_phoneme(word: str) -> str:
    raw = g2p_en_model(word)
    phones = [p for p in raw if p and p != " " and p != "<unk>"]
    ipa = arpabet_to_ipa_sequence(phones)
    return ipa if ipa else g2p_id_phoneme(word)


def rumus_a_singkatan(word: str) -> str:
    ejaan = "".join(ALPHABET_EJA_MAP.get(ch, ch) for ch in word.lower() if ch.isalpha())
    if not ejaan:
        return g2p_id_phoneme(word)
    return g2p_id_phoneme(ejaan)


def rumus_bc_disemvowelling(word: str) -> str:
    alpha_only = "".join(ch for ch in word.lower() if ch.isalpha())
    if not alpha_only:
        return DISMV_SHORT_PATTERN
    if len(alpha_only) <= 3:
        return DISMV_SHORT_PATTERN
    return DISMV_LONG_PATTERN


def _normalize_label(label: str) -> str:
    return label.upper().strip()


LABELS_ASING = {"B-ASG", "I-ASG"}
LABELS_SINGKATAN = {"B-SKT", "I-SKT"}
LABELS_DISEMVOWEL = {"B-DSM"}
LABELS_TYPO = {"B-TYP"}
LABELS_VALID = {"O", *LABELS_ASING, *LABELS_SINGKATAN, *LABELS_DISEMVOWEL, *LABELS_TYPO}


def layer2a_2b_switchboard(token: str, label: str) -> str:
    norm_label = _normalize_label(label)

    if norm_label == "O":
        # Layer 2b (jalur baku): label O -> g2p-id
        return g2p_id_phoneme(token)

    if norm_label in LABELS_ASING:
        # Layer 2b (jalur asing): bahasa asing -> g2p biasa (g2p-en)
        return g2p_foreign_phoneme(token)

    if norm_label in LABELS_SINGKATAN:
        # Layer 2a: Rumus A untuk singkatan
        return rumus_a_singkatan(token)

    if norm_label in LABELS_DISEMVOWEL:
        # Layer 2a: Rumus B/C untuk disemvowelling
        return rumus_bc_disemvowelling(token)

    if norm_label in LABELS_TYPO:
        # Layer 2a: typo diarahkan ke g2p-id sebagai OOV fallback
        return g2p_id_phoneme(token)

    raise ValueError(
        f"Label tidak dikenali: {label!r}. Label yang didukung: {sorted(LABELS_VALID)}"
    )


def run_layer2_from_predictions(predictions: list[TokenLabel]) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    for item in predictions:
        phoneme = layer2a_2b_switchboard(item.token, item.label)
        rows.append({
            "token": item.token,
            "label": item.label,
            "score": round(item.score, 6),
            "phoneme": phoneme,
        })
    return rows


def run_layer2_batch(batch_predictions: list[list[TokenLabel]]) -> list[list[dict[str, object]]]:
    return [run_layer2_from_predictions(preds) for preds in batch_predictions]

In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any


@dataclass(frozen=True)
class TokenLabel:
    token: str
    label: str
    score: float


def _validate_model_dir(model_dir: Path) -> None:
    required_files = ("config.json", "tokenizer.json", "tokenizer_config.json")
    missing_required = [name for name in required_files if not (model_dir / name).exists()]
    if missing_required:
        raise FileNotFoundError(
            f"Model directory tidak lengkap: {model_dir}. "
            f"File wajib belum ada: {', '.join(missing_required)}."
        )

    has_weights = any(
        (model_dir / filename).exists()
        for filename in (
            "pytorch_model.bin",
            "model.safetensors",
            "tf_model.h5",
            "model.safetensors.index.json",
            "pytorch_model.bin.index.json",
        )
    )
    if not has_weights:
        raise FileNotFoundError(
            f"Model weights tidak ditemukan di {model_dir}. "
            "Simpan salah satu file berikut: "
            "pytorch_model.bin / model.safetensors / tf_model.h5 "
            "(atau index file untuk model yang di-shard)."
        )


def _import_runtime_deps() -> tuple[Any, Any, Any]:
    try:
        import torch  # type: ignore
        from transformers import AutoModelForTokenClassification, AutoTokenizer  # type: ignore
    except ModuleNotFoundError as err:
        raise ModuleNotFoundError(
            "Dependensi belum lengkap. Install dulu: python -m pip install torch transformers"
        ) from err
    return torch, AutoTokenizer, AutoModelForTokenClassification


def load_layer1_model(model_dir: Path):
    _validate_model_dir(model_dir)
    torch, AutoTokenizer, AutoModelForTokenClassification = _import_runtime_deps()
    tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
    model = AutoModelForTokenClassification.from_pretrained(str(model_dir))
    if not torch.cuda.is_available():
        raise RuntimeError("GPU CUDA tidak terdeteksi. Pastikan Colab runtime memakai GPU.")
    device = torch.device("cuda")
    model.to(device)
    model.eval()
    return tokenizer, model, torch, device


def predict_labels(
    text: str,
    tokenizer,
    model,
    torch,
    max_length: int = 256,
) -> list[TokenLabel]:
    words = text.strip().split()
    if not words:
        return []

    encoded = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
    )
    model_inputs = {key: value.to(model.device) for key, value in encoded.items()}
    with torch.inference_mode():
        output = model(**model_inputs)

    probs = torch.softmax(output.logits, dim=-1)
    pred_ids = torch.argmax(output.logits, dim=-1)[0].tolist()
    word_ids = encoded.word_ids(batch_index=0)
    id2label = model.config.id2label
    if not id2label:
        raise ValueError("id2label tidak ditemukan pada config model.")

    results: list[TokenLabel] = []
    seen_word_index = set()
    for token_idx, word_idx in enumerate(word_ids):
        if word_idx is None or word_idx in seen_word_index:
            continue
        seen_word_index.add(word_idx)
        pred_id = pred_ids[token_idx]
        label = id2label.get(pred_id, str(pred_id))
        confidence = float(probs[0, token_idx, pred_id].item())
        results.append(TokenLabel(token=words[word_idx], label=label, score=confidence))

    return results


def predict_labels_batch(
    texts: list[str],
    tokenizer,
    model,
    torch,
    max_length: int = 256,
    batch_size: int = 16,
) -> list[list[TokenLabel]]:
    if not texts:
        return []

    cleaned_texts = [text.strip() for text in texts]
    words_per_text = [text.split() for text in cleaned_texts]
    all_results: list[list[TokenLabel]] = [[] for _ in cleaned_texts]
    id2label = model.config.id2label
    if not id2label:
        raise ValueError("id2label tidak ditemukan pada config model.")

    for start in range(0, len(cleaned_texts), batch_size):
        end = min(start + batch_size, len(cleaned_texts))
        batch_words = words_per_text[start:end]
        encoded = tokenizer(
            batch_words,
            is_split_into_words=True,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=True,
        )
        model_inputs = {key: value.to(model.device) for key, value in encoded.items()}
        with torch.inference_mode():
            output = model(**model_inputs)

        probs = torch.softmax(output.logits, dim=-1)
        pred_ids = torch.argmax(output.logits, dim=-1).tolist()

        for local_idx, words in enumerate(batch_words):
            if not words:
                continue
            word_ids = encoded.word_ids(batch_index=local_idx)
            seen_word_index = set()
            row_results: list[TokenLabel] = []
            for token_idx, word_idx in enumerate(word_ids):
                if word_idx is None or word_idx in seen_word_index:
                    continue
                seen_word_index.add(word_idx)
                pred_id = pred_ids[local_idx][token_idx]
                label = id2label.get(pred_id, str(pred_id))
                confidence = float(probs[local_idx, token_idx, pred_id].item())
                row_results.append(TokenLabel(token=words[word_idx], label=label, score=confidence))
            all_results[start + local_idx] = row_results

    return all_results


def _print_results(text: str, predictions: list[TokenLabel]) -> None:
    print(f"Input: {text}")
    if not predictions:
        print("Tidak ada token yang bisa diprediksi.")
        return
    for item in predictions:
        print(f"{item.token}\t{item.label}\t{item.score:.4f}")


def _to_json_record(text: str, predictions: list[TokenLabel]) -> str:
    payload = {
        "text": text,
        "tokens": [
            {"token": item.token, "label": item.label, "score": round(item.score, 6)}
            for item in predictions
        ],
    }
    return json.dumps(payload, ensure_ascii=False)

In [ ]:
# Mount Google Drive (wajib untuk model di MyDrive).
from google.colab import drive
drive.mount('/content/drive')

MODEL_DIR = Path("/content/drive/MyDrive/Models/indobert-pantun-ner")
MAX_LENGTH = 64
BATCH_SIZE = 32

tokenizer, model, torch, device = load_layer1_model(MODEL_DIR)
print("Model siap dipakai dari:", MODEL_DIR)
print("Device:", device)

In [ ]:
text = "brgkt ke spbu naik motor dpr"
preds = predict_labels(
    text,
    tokenizer=tokenizer,
    model=model,
    torch=torch,
    max_length=MAX_LENGTH,
)
_print_results(text, preds)

In [ ]:
# Layer 2a/2b untuk hasil single inference.
layer2_rows = run_layer2_from_predictions(preds)
for row in layer2_rows:
    print(f"{row['token']}\t{row['label']}\t{row['phoneme']}")

In [ ]:
# Inferensi batch dari file (1 baris = 1 input), lalu simpan JSONL.
INPUT_FILE = Path("/content/drive/MyDrive/Models/pantun_gaul_for_llm_batch_01.txt")
OUTPUT_JSONL = Path("/content/drive/MyDrive/Models/layer1_predictions_batch01.jsonl")

lines = [
    line.strip()
    for line in INPUT_FILE.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

batched_predictions = predict_labels_batch(
    lines,
    tokenizer=tokenizer,
    model=model,
    torch=torch,
    max_length=MAX_LENGTH,
    batch_size=max(1, BATCH_SIZE),
)

jsonl_rows = []
for idx, (line, preds) in enumerate(zip(lines, batched_predictions), start=1):
    print(f"\n=== Baris {idx} ===")
    _print_results(line, preds)
    jsonl_rows.append(_to_json_record(line, preds))

OUTPUT_JSONL.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_JSONL.write_text("\n".join(jsonl_rows) + "\n", encoding="utf-8")
print(f"\nHasil JSONL tersimpan di: {OUTPUT_JSONL}")

In [ ]:
# Layer 2a/2b untuk hasil batch; output JSONL berisi token, label, score, fonem.
OUTPUT_LAYER2_JSONL = Path("/content/drive/MyDrive/Models/layer2a2b_phonemes_batch01.jsonl")

layer2_batch = run_layer2_batch(batched_predictions)
layer2_jsonl_rows = []
for line, layer2_rows in zip(lines, layer2_batch):
    layer2_jsonl_rows.append(json.dumps({
        "text": line,
        "tokens": layer2_rows,
    }, ensure_ascii=False))

OUTPUT_LAYER2_JSONL.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_LAYER2_JSONL.write_text("\n".join(layer2_jsonl_rows) + "\n", encoding="utf-8")
print("Hasil Layer 2a/2b tersimpan di:", OUTPUT_LAYER2_JSONL)